In [29]:
import os
import re
import pickle
import pandas as pd

logs_dir = "logs"
output_dir = "out"

def get_runtime(run_name):

    out_file = os.path.join(output_dir, f"{run_name}.out")

    if not os.path.exists(out_file):

        return None

    with open(out_file) as f:

        text = f.read()

    m = re.search(r"Time:\s*(.*?)\n", text)

    if m:

        runtime = m.group(1)

        return runtime

    return None

def get_memory(run_name):

    log_file = os.path.join(logs_dir, f"{run_name}.log")

    if not os.path.exists(log_file):

        return None

    with open(log_file) as f:

        text = f.read()

    m = re.search(

    r"Memory\s*\(MB\)\s*:\s*([\d\.]+)",text)

    if m:

        memory = float(m.group(1))

        return memory

    return None



In [30]:
rows = []

for file in os.listdir("./pkl"):
    if not file.endswith(".pkl"):

        continue

    with open(f"./pkl/{file}", "rb") as f:

        graph, cols = pickle.load(f)

    matrix = graph.graph

    directed = 0

    circle_arrow = 0

    bidirected = 0

    for i in range(len(matrix)):

        for j in range(i + 1, len(matrix)):

            a = matrix[i, j]

            b = matrix[j, i]


            if (a,b) == (-1,1) or (a,b) == (1,-1):

                directed += 1

            elif (a,b) == (2,1) or (a,b) == (1,2):

                circle_arrow += 1

            elif (a,b) == (1,1):

                bidirected += 1
    run_name = file.replace(".pkl", "")

    runtime = get_runtime(run_name) 

    memory = get_memory(run_name)   

    rows.append({

        "run": run_name,

        "algorithm": "FCI" if file.startswith("FCI") else "PC",

        "alpha":

            0.01 if "_a01_" in file else

            0.05 if "_a05_" in file else

            0.10,

        "directed_edges": directed,

        "circle_arrow_edges": circle_arrow,

        "bidirected_edges": bidirected,

        "total_edges": directed + circle_arrow + bidirected,

        "nodes": len(cols),

        "runtime": runtime,
        
        "memory": memory

    })

df = pd.DataFrame(rows)

df.sort_values("total_edges")
    

,run,algorithm,alpha,directed_edges,circle_arrow_edges,bidirected_edges,total_edges,nodes,runtime,memory
5,FCI_fisherz_simple_a01_5k_d-1,FCI,0.01,54,6,18,78,25,753.2s (0.21hrs),842.0
16,FCI_fisherz_simple_a01_5k_d4,FCI,0.01,52,8,18,78,25,1138.5s (0.32hrs),1206.0
14,FCI_fisherz_simple_a01_5k_d3,FCI,0.01,54,4,21,79,25,2973.3s (0.83hrs),2209.0
23,FCI_fisherz_simple_a01_10k_d3,FCI,0.01,62,9,26,97,25,4700.0s (1.31hrs),4708.0
18,FCI_fisherz_simple_a05_5k_d-1,FCI,0.05,61,8,28,97,25,2735.6s (0.76hrs),2727.0
1,FCI_fisherz_simple_a05_5k_d4,FCI,0.05,63,8,28,99,25,5893.2s (1.64hrs),5617.0
3,FCI_fisherz_simple_a05_5k_d3,FCI,0.05,63,7,29,99,25,8017.0s (2.23hrs),8061.0
20,FCI_fisherz_simple_a01_10k_d-1,FCI,0.01,63,8,30,101,25,3245.1s (0.90hrs),3656.0
19,FCI_fisherz_simple_a01_10k_d4,FCI,0.01,63,8,30,101,25,4433.4s (1.23hrs),3793.0
9,FCI_fisherz_simple_a10_5k_d4,FCI,0.10,67,4,35,106,25,59513.7s (16.53hrs),8117.0
